In [1]:
# ============================================================
# GPT-2 LEGAL DOMAIN ADAPTATION
# FAST / T4-FRIENDLY EXPERIMENT
#
# Model       : GPT-2 Small (124M)
# Objective   : Causal Language Modeling
# Dataset     : Indian Legal Corpus
# GPU         : Kaggle Tesla T4
#
# IMPORTANT:
# This is CONTINUED PRETRAINING / DAPT.
# GPT-2 is NOT trained from scratch.
# ============================================================


In [2]:
!pip install -q -U \
    transformers \
    datasets \
    accelerate \
    evaluate \
    huggingface_hub

import os
import gc
import math
import random
import shutil
import zipfile
import time

import numpy as np
import pandas as pd
import torch

from pathlib import Path

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    set_seed
)

from huggingface_hub import hf_hub_download

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 94.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [3]:
SEED = 61

set_seed(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


print("=" * 70)
print("ENVIRONMENT")
print("=" * 70)

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU             :", torch.cuda.get_device_name(0))

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3
    )

    print(
        "GPU memory      :",
        round(gpu_memory, 2),
        "GB"
    )


ENVIRONMENT
PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
GPU memory      : 14.56 GB


In [4]:
MODEL_NAME = "openai-community/gpt2"

# Standard GPT-2 context length
BLOCK_SIZE = 1024

In [5]:
MAX_TRAIN_BLOCKS = 100_000

# Full validation was ~36k blocks and took ~44 minutes.
# 2,000 blocks gives a stable, much faster evaluation.
MAX_VALID_BLOCKS = 2_000

In [6]:
NUM_EPOCHS = 1

LEARNING_RATE = 2e-5

WEIGHT_DECAY = 0.01


PER_DEVICE_BATCH_SIZE = 2

GRADIENT_ACCUMULATION_STEPS = 4

In [7]:
OUTPUT_DIR = "/kaggle/working/gpt2_legal_dapt"

FINAL_MODEL_DIR = (
    "/kaggle/working/"
    "gpt2_legal_dapt_final"
)

RESULTS_PATH = (
    "/kaggle/working/"
    "gpt2_legal_dapt_results.csv"
)

In [8]:
print("\n" + "=" * 70)
print("MODEL CONFIGURATION")
print("=" * 70)

print("Model                     :", MODEL_NAME)
print("Block size                :", BLOCK_SIZE)
print("Epochs                    :", NUM_EPOCHS)
print("Learning rate             :", LEARNING_RATE)
print("Weight decay              :", WEIGHT_DECAY)
print("Per-device batch size     :", PER_DEVICE_BATCH_SIZE)
print(
    "Gradient accumulation     :",
    GRADIENT_ACCUMULATION_STEPS
)

print(
    "Effective batch size      :",
    PER_DEVICE_BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS
)

print(
    "Maximum training blocks  :",
    MAX_TRAIN_BLOCKS
)

print(
    "Maximum validation blocks:",
    MAX_VALID_BLOCKS
)



print("\n" + "=" * 70)
print("DOWNLOADING ORIGINAL LEGAL DATASET")
print("=" * 70)


HF_REPO = "L-NLProc/InLegalLlama-training-data"

HF_FILENAME = "For CPT/Train_data.zip"

DATA_DIR = "/kaggle/working/legal_data"

ZIP_PATH = "/kaggle/working/Train_data.zip"


os.makedirs(DATA_DIR, exist_ok=True)



MODEL CONFIGURATION
Model                     : openai-community/gpt2
Block size                : 1024
Epochs                    : 1
Learning rate             : 2e-05
Weight decay              : 0.01
Per-device batch size     : 2
Gradient accumulation     : 4
Effective batch size      : 8
Maximum training blocks  : 100000
Maximum validation blocks: 2000

DOWNLOADING ORIGINAL LEGAL DATASET


In [9]:

try:

    downloaded_path = hf_hub_download(
        repo_id=HF_REPO,
        filename=HF_FILENAME,
        repo_type="dataset"
    )

    print("Downloaded publicly:")
    print(downloaded_path)

except Exception as e:

    print("\nPublic download failed.")
    print("Trying Hugging Face token...")

    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()

    HF_TOKEN = user_secrets.get_secret(
        "HF-MTP-SLM"
    )

    downloaded_path = hf_hub_download(
        repo_id=HF_REPO,
        filename=HF_FILENAME,
        repo_type="dataset",
        token=HF_TOKEN
    )

    print("Downloaded successfully using HF token.")
    print(downloaded_path)



shutil.copy(
    downloaded_path,
    ZIP_PATH
)

print("\nZIP file:")
print(ZIP_PATH)


print("\n" + "=" * 70)
print("EXTRACTING DATASET")
print("=" * 70)

with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as zip_ref:

    zip_ref.extractall(DATA_DIR)


print("Extracted to:")
print(DATA_DIR)


print("\n" + "=" * 70)
print("SEARCHING FOR CSV")
print("=" * 70)


csv_files = list(
    Path(DATA_DIR).rglob("*.csv")
)


print("CSV files found:")

for f in csv_files:

    print(" -", f)


if len(csv_files) == 0:

    raise FileNotFoundError(
        "No CSV file found inside extracted dataset."
    )


CSV_PATH = (
    "/kaggle/working/legal_data/"
    "Train_data/"
    "train_for_llama2_cpt_SCI_38k_HC_100k_from_1800_2020.csv"
)


if not os.path.exists(CSV_PATH):

    print(
        "\nSpecified CSV path not found."
    )

    print(
        "Using first CSV found instead."
    )

    CSV_PATH = str(
        csv_files[0]
    )


print("\nUsing:")
print(CSV_PATH)



Public download failed.
Trying Hugging Face token...


For CPT/Train_data.zip: reconstructing file:   0%|          |  0.00B /  518MB            

For CPT/Train_data.zip: downloading bytes:           |  0.00B            

Downloaded successfully using HF token.
/root/.cache/huggingface/hub/datasets--L-NLProc--InLegalLlama-training-data/snapshots/91453101ebc33c4e49c9eb7376235aca1f18065b/For CPT/Train_data.zip

ZIP file:
/kaggle/working/Train_data.zip

EXTRACTING DATASET
Extracted to:
/kaggle/working/legal_data

SEARCHING FOR CSV
CSV files found:
 - /kaggle/working/legal_data/Train_data/train_for_llama2_cpt_SCI_38k_HC_100k_from_1800_2020.csv

Using:
/kaggle/working/legal_data/Train_data/train_for_llama2_cpt_SCI_38k_HC_100k_from_1800_2020.csv


In [10]:
print("\n" + "=" * 70)
print("LOADING RAW LEGAL DATA")
print("=" * 70)


df = pd.read_csv(
    CSV_PATH
)


print(
    "Original shape:",
    df.shape
)

print(
    "Columns:",
    df.columns.tolist()
)


if "text" not in df.columns:

    raise ValueError(
        "'text' column not found."
    )


# Keep ONLY text

df = df[
    ["text"]
].copy()


# Remove missing values

df["text"] = df[
    "text"
].astype(str)


df = df[
    df["text"].str.strip() != ""
]


print(
    "\nAfter removing empty documents:"
)

print(
    "Shape:",
    df.shape
)



LOADING RAW LEGAL DATA
Original shape: (138321, 1)
Columns: ['text']

After removing empty documents:
Shape: (138321, 1)


In [11]:
print("\n" + "=" * 70)
print("REMOVING EXACT DUPLICATES")
print("=" * 70)


before_duplicates = len(df)


df = df.drop_duplicates(
    subset=["text"],
    keep="first"
).reset_index(
    drop=True
)


after_duplicates = len(df)


duplicates_removed = (
    before_duplicates
    - after_duplicates
)


print(
    "Documents before :",
    before_duplicates
)

print(
    "Duplicates removed:",
    duplicates_removed
)

print(
    "Documents after  :",
    after_duplicates
)

print(
    "Duplicate percentage:",
    round(
        duplicates_removed
        / before_duplicates
        * 100,
        4
    ),
    "%"
)

print("\n" + "=" * 70)
print("TRAIN / VALIDATION SPLIT")
print("=" * 70)

dataset = Dataset.from_pandas(
    df,
    preserve_index=False
)


split_dataset = dataset.train_test_split(
    test_size=0.10,
    seed=SEED
)


train_dataset = split_dataset[
    "train"
]

valid_dataset = split_dataset[
    "test"
]


print(
    "Training documents  :",
    len(train_dataset)
)

print(
    "Validation documents:",
    len(valid_dataset)
)


REMOVING EXACT DUPLICATES
Documents before : 138321
Duplicates removed: 4665
Documents after  : 133656
Duplicate percentage: 3.3726 %

TRAIN / VALIDATION SPLIT
Training documents  : 120290
Validation documents: 13366


In [12]:
print("\n" + "=" * 70)
print("LOADING GPT-2 TOKENIZER")
print("=" * 70)


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


# GPT-2 does not have a separate PAD token.

if tokenizer.pad_token is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )


print(
    "Tokenizer:",
    MODEL_NAME
)

print(
    "Vocabulary size:",
    len(tokenizer)
)

print(
    "EOS token:",
    tokenizer.eos_token
)

print(
    "EOS ID:",
    tokenizer.eos_token_id
)

print(
    "PAD token:",
    tokenizer.pad_token
)

print(
    "PAD ID:",
    tokenizer.pad_token_id
)



print("\n" + "=" * 70)
print("TOKENIZING DATA")
print("=" * 70)


def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        add_special_tokens=True,
        truncation=False
    )



LOADING GPT-2 TOKENIZER


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer: openai-community/gpt2
Vocabulary size: 50257
EOS token: <|endoftext|>
EOS ID: 50256
PAD token: <|endoftext|>
PAD ID: 50256

TOKENIZING DATA


In [13]:
tokenized_train = train_dataset.map(

    tokenize_function,

    batched=True,

    num_proc=2,

    remove_columns=["text"],

    desc="Tokenizing training documents"
)


print(
    "Training tokenization complete."
)


tokenized_valid = valid_dataset.map(

    tokenize_function,

    batched=True,

    num_proc=2,

    remove_columns=["text"],

    desc="Tokenizing validation documents"
)


print(
    "Validation tokenization complete."
)


Tokenizing training documents (num_proc=2):   0%|          | 0/120290 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (8549 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1273 > 1024). Running this sequence through the model will result in indexing errors


Training tokenization complete.


Tokenizing validation documents (num_proc=2):   0%|          | 0/13366 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2231 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1369 > 1024). Running this sequence through the model will result in indexing errors


Validation tokenization complete.


In [14]:
print("\n" + "=" * 70)
print("CREATING 1024-TOKEN BLOCKS")
print("=" * 70)


def group_texts(examples):

    concatenated = {}

    for key in examples.keys():

        concatenated[key] = sum(
            examples[key],
            []
        )


    total_length = len(
        concatenated["input_ids"]
    )


    # Drop incomplete final block.

    total_length = (
        total_length
        // BLOCK_SIZE
    ) * BLOCK_SIZE


    result = {

        key: [

            tokens[
                i:i + BLOCK_SIZE
            ]

            for i in range(
                0,
                total_length,
                BLOCK_SIZE
            )
        ]

        for key, tokens
        in concatenated.items()
    }


    return result


lm_train = tokenized_train.map(

    group_texts,

    batched=True,

    batch_size=1000,

    num_proc=2,

    desc="Grouping training tokens"
)


print(
    "\nFull training blocks:",
    len(lm_train)
)



lm_valid = tokenized_valid.map(

    group_texts,

    batched=True,

    batch_size=1000,

    num_proc=2,

    desc="Grouping validation tokens"
)


print(
    "Full validation blocks:",
    len(lm_valid)
)



CREATING 1024-TOKEN BLOCKS


Grouping training tokens (num_proc=2):   0%|          | 0/120290 [00:00<?, ? examples/s]


Full training blocks: 322370


Grouping validation tokens (num_proc=2):   0%|          | 0/13366 [00:00<?, ? examples/s]

Full validation blocks: 36105


In [15]:
print("\n" + "=" * 70)
print("SELECTING EXPERIMENT DATA")
print("=" * 70)

# Shuffle BEFORE selecting.
#
# Otherwise we would only train on the first 100k blocks.

lm_train = lm_train.shuffle(
    seed=SEED
)

train_blocks_to_use = min(
    MAX_TRAIN_BLOCKS,
    len(lm_train)
)


lm_train = lm_train.select(
    range(
        train_blocks_to_use
    )
)


valid_blocks_to_use = min(
    MAX_VALID_BLOCKS,
    len(lm_valid)
)


lm_valid = lm_valid.select(
    range(
        valid_blocks_to_use
    )
)


print(
    "Training blocks used   :",
    len(lm_train)
)

print(
    "Validation blocks used :",
    len(lm_valid)
)


print(
    "Training tokens used   :",
    len(lm_train)
    * BLOCK_SIZE
)

print(
    "Validation tokens used :",
    len(lm_valid)
    * BLOCK_SIZE
)



SELECTING EXPERIMENT DATA
Training blocks used   : 100000
Validation blocks used : 2000
Training tokens used   : 102400000
Validation tokens used : 2048000


In [16]:
print("\n" + "=" * 70)
print("SAMPLE TOKENIZATION CHECK")
print("=" * 70)


sample_ids = lm_train[0][
    "input_ids"
]


print(
    "Number of tokens:",
    len(sample_ids)
)


print(
    "\nFirst 50 token IDs:"
)

print(
    sample_ids[:50]
)


print(
    "\nDecoded beginning:"
)

print(
    tokenizer.decode(
        sample_ids[:200]
    )
)



SAMPLE TOKENIZATION CHECK
Number of tokens: 1024

First 50 token IDs:
[8978, 257, 4081, 7664, 393, 523, 28805, 326, 340, 318, 32848, 290, 7787, 284, 1011, 1243, 284, 511, 3288, 383, 49440, 3078, 373, 33595, 1512, 287, 8296, 326, 4414, 286, 4719, 750, 407, 8084, 4078, 39979, 286, 262, 5371, 416, 12600, 284, 37262, 942, 393, 39980, 4135, 18506, 13, 818]

Decoded beginning:
 reaching a firm conclusion or so timed that it is hesitant and afraid to take things to their natural The Apex Court was emphatic in recording that benefit of doubt did not warrant acquittal of the accused by resort to conjectures or fanciful considerations.In arriving at the conclusion about the guilt of the accused charged with the commission of a crime, the court has to judge the by the yardstick of its worth and the animus of and the final analysis would depend in a case upon its own facts.Lakshmi Singh and ors.v. State of Bihar (supra) has been cited by the in support of the that fouler the crime higher is the s

In [17]:
print("\n" + "=" * 70)
print("CREATING CAUSAL LM DATA COLLATOR")
print("=" * 70)


data_collator = (
    DataCollatorForLanguageModeling(

        tokenizer=tokenizer,

        mlm=False
    )
)


print(
    "MLM:",
    False
)

print(
    "Objective:",
    "Causal Language Modeling"
)



CREATING CAUSAL LM DATA COLLATOR
MLM: False
Objective: Causal Language Modeling


In [18]:


print("\n" + "=" * 70)
print("LOADING PRETRAINED GPT-2")
print("=" * 70)


model = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL_NAME
    )
)


# Correct padding

model.config.pad_token_id = (
    tokenizer.pad_token_id
)


print(
    "Model loaded successfully."
)


num_parameters = sum(
    p.numel()
    for p in model.parameters()
)


trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    "\nTotal parameters     :",
    f"{num_parameters:,}"
)

print(
    "Total parameters (M) :",
    f"{num_parameters / 1e6:.2f}M"
)

print(
    "Trainable parameters :",
    f"{trainable_parameters:,}"
)

if torch.cuda.is_available():

    print("\n" + "=" * 70)
    print("GPU MEMORY")
    print("=" * 70)


    allocated = (
        torch.cuda.memory_allocated()
        / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved()
        / 1024**3
    )


    print(
        "Allocated:",
        round(allocated, 2),
        "GB"
    )

    print(
        "Reserved :",
        round(reserved, 2),
        "GB"
    )


LOADING PRETRAINED GPT-2


model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully.

Total parameters     : 124,439,808
Total parameters (M) : 124.44M
Trainable parameters : 124,439,808

GPU MEMORY
Allocated: 0.0 GB
Reserved : 0.0 GB


In [19]:
print("\n" + "=" * 70)
print("SETTING TRAINING PARAMETERS")
print("=" * 70)


training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,

    learning_rate=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY,

    warmup_steps=100,


    # Scheduler

    lr_scheduler_type="cosine",

    per_device_train_batch_size=(
        PER_DEVICE_BATCH_SIZE
    ),

    per_device_eval_batch_size=(
        PER_DEVICE_BATCH_SIZE
    ),

    gradient_accumulation_steps=(
        GRADIENT_ACCUMULATION_STEPS
    ),


    # --------------------------------------------------------
    # EVALUATION
    # --------------------------------------------------------
    #
    # IMPORTANT:
    #
    # We do NOT evaluate every few steps.
    #
    # We evaluate only at the end of the epoch.
    #
    # This saves a lot of time.
    # --------------------------------------------------------

    eval_strategy="epoch",


    # --------------------------------------------------------
    # SAVING
    # --------------------------------------------------------

    save_strategy="epoch",

    save_total_limit=1,


    # We want the best model according to validation loss.

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,


    # --------------------------------------------------------
    # LOGGING
    # --------------------------------------------------------

    logging_strategy="steps",

    logging_steps=100,


    # --------------------------------------------------------
    # MIXED PRECISION
    # --------------------------------------------------------

    fp16=torch.cuda.is_available(),


    # --------------------------------------------------------
    # MEMORY
    # --------------------------------------------------------
    #
    # Gradient checkpointing reduces memory usage.
    #
    # We keep it enabled so batch size 2 remains safe on T4.
    # --------------------------------------------------------

    gradient_checkpointing=True,

    dataloader_num_workers=2,

    dataloader_pin_memory=True,

    seed=SEED,

    data_seed=SEED,

    report_to="none",

    remove_unused_columns=True
)



SETTING TRAINING PARAMETERS


In [20]:
print("\n" + "=" * 70)
print("CREATING TRAINER")
print("=" * 70)


trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=lm_train,

    eval_dataset=lm_valid,

    data_collator=data_collator,

    processing_class=tokenizer
)


print(
    "Trainer created successfully."
)



CREATING TRAINER
Trainer created successfully.


In [21]:
print("\n" + "=" * 70)
print("BASELINE EVALUATION — BEFORE LEGAL DAPT")
print("=" * 70)


print(
    "Evaluating GPT-2 on the legal validation subset..."
)


baseline_start = time.time()


initial_results = trainer.evaluate()


baseline_time = (
    time.time()
    - baseline_start
)


initial_loss = (
    initial_results["eval_loss"]
)


try:

    initial_perplexity = math.exp(
        initial_loss
    )

except OverflowError:

    initial_perplexity = float(
        "inf"
    )


print("\n" + "-" * 70)
print("INITIAL RESULTS")
print("-" * 70)


print(
    "Validation loss       :",
    f"{initial_loss:.4f}"
)

print(
    "Validation perplexity :",
    f"{initial_perplexity:.4f}"
)

print(
    "Evaluation time       :",
    f"{baseline_time / 60:.2f} minutes"
)



BASELINE EVALUATION — BEFORE LEGAL DAPT
Evaluating GPT-2 on the legal validation subset...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Training Loss,Validation Loss,Epoch
No log,3.449754,0



----------------------------------------------------------------------
INITIAL RESULTS
----------------------------------------------------------------------
Validation loss       : 3.4498
Validation perplexity : 31.4926
Evaluation time       : 2.17 minutes


In [22]:
gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


In [23]:
print("\n" + "=" * 70)
print("STARTING GPT-2 DOMAIN-ADAPTIVE PRETRAINING")
print("=" * 70)


print()
print(
    "PRETRAINED GPT-2:"
)

print(
    "  Yes"
)

print()
print(
    "Training objective:"
)

print(
    "  Continued pretraining / DAPT"
)

print()
print(
    "Training blocks:"
)

print(
    f"  {len(lm_train):,}"
)

print()
print(
    "Training tokens:"
)

print(
    f"  {len(lm_train) * BLOCK_SIZE:,}"
)

print()
print(
    "Expected optimizer steps:"
)

optimizer_steps = math.ceil(
    len(lm_train)
    / (
        PER_DEVICE_BATCH_SIZE
        * GRADIENT_ACCUMULATION_STEPS
    )
)

print(
    f"  {optimizer_steps:,}"
)

print()
print(
    "Starting training..."
)

print()


training_start = time.time()


train_result = trainer.train()


training_time = (
    time.time()
    - training_start
)


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.



STARTING GPT-2 DOMAIN-ADAPTIVE PRETRAINING

PRETRAINED GPT-2:
  Yes

Training objective:
  Continued pretraining / DAPT

Training blocks:
  100,000

Training tokens:
  102,400,000

Expected optimizer steps:
  12,500

Starting training...



Epoch,Training Loss,Validation Loss
1,2.862840,2.789252


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


In [24]:
print("\n" + "=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)


print(
    "\nTraining time:"
)

print(
    f"{training_time / 3600:.2f} hours"
)


print(
    f"{training_time / 60:.2f} minutes"
)


print(
    "\nTraining metrics:"
)


for key, value in (
    train_result.metrics.items()
):

    if isinstance(
        value,
        float
    ):

        print(
            f"{key}: {value:.6f}"
        )

    else:

        print(
            f"{key}: {value}"
        )



TRAINING COMPLETED

Training time:
6.14 hours
368.51 minutes

Training metrics:
train_runtime: 22110.061100
train_samples_per_second: 4.523000
train_steps_per_second: 0.283000
total_flos: 52258406400000000.000000
train_loss: 2.935969
epoch: 1.000000


In [25]:
print("\n" + "=" * 70)
print("FINAL VALIDATION")
print("=" * 70)


final_eval_start = time.time()


final_results = (
    trainer.evaluate()
)


final_eval_time = (
    time.time()
    - final_eval_start
)


final_loss = (
    final_results["eval_loss"]
)


try:

    final_perplexity = math.exp(
        final_loss
    )

except OverflowError:

    final_perplexity = float(
        "inf"
    )


print(
    "Final validation loss:",
    f"{final_loss:.4f}"
)

print(
    "Final validation perplexity:",
    f"{final_perplexity:.4f}"
)

print(
    "Final evaluation time:",
    f"{final_eval_time / 60:.2f} minutes"
)



FINAL VALIDATION


Training Loss,Validation Loss,Epoch
2.862840,2.789252,1


Final validation loss: 2.7893
Final validation perplexity: 16.2688
Final evaluation time: 2.26 minutes


In [26]:
loss_improvement = (
    initial_loss
    - final_loss
)


ppl_improvement = (
    initial_perplexity
    - final_perplexity
)


loss_improvement_percent = (

    loss_improvement
    / initial_loss
    * 100
)


ppl_improvement_percent = (

    ppl_improvement
    / initial_perplexity
    * 100
)


print("\n" + "=" * 70)
print("IMPROVEMENT FROM DOMAIN ADAPTATION")
print("=" * 70)


print(
    "\nLoss improvement:"
)

print(
    f"{loss_improvement:.4f}"
)


print(
    "Loss improvement (%):"
)

print(
    f"{loss_improvement_percent:.2f}%"
)


print(
    "\nPerplexity improvement:"
)

print(
    f"{ppl_improvement:.4f}"
)


print(
    "Perplexity improvement (%):"
)

print(
    f"{ppl_improvement_percent:.2f}%"
)


print("\n" + "=" * 70)
print("SAVING FINAL MODEL")
print("=" * 70)


trainer.save_model(
    FINAL_MODEL_DIR
)


tokenizer.save_pretrained(
    FINAL_MODEL_DIR
)


print(
    "\nModel saved to:"
)

print(
    FINAL_MODEL_DIR
)




IMPROVEMENT FROM DOMAIN ADAPTATION

Loss improvement:
0.6605
Loss improvement (%):
19.15%

Perplexity improvement:
15.2238
Perplexity improvement (%):
48.34%

SAVING FINAL MODEL


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to:
/kaggle/working/gpt2_legal_dapt_final


In [27]:
results_df = pd.DataFrame({

    "model": [
        "GPT-2"
    ],

    "architecture": [
        "Transformer"
    ],

    "initialization": [
        "Pretrained"
    ],

    "objective": [
        "Causal Language Modeling"
    ],

    "context_length": [
        BLOCK_SIZE
    ],

    "epochs": [
        NUM_EPOCHS
    ],

    "learning_rate": [
        LEARNING_RATE
    ],

    "weight_decay": [
        WEIGHT_DECAY
    ],

    "per_device_batch_size": [
        PER_DEVICE_BATCH_SIZE
    ],

    "gradient_accumulation_steps": [
        GRADIENT_ACCUMULATION_STEPS
    ],

    "effective_batch_size": [
        PER_DEVICE_BATCH_SIZE
        * GRADIENT_ACCUMULATION_STEPS
    ],

    "train_documents": [
        len(train_dataset)
    ],

    "validation_documents": [
        len(valid_dataset)
    ],

    "full_train_blocks": [
        len(tokenized_train)
    ],

    "full_validation_blocks": [
        len(tokenized_valid)
    ],

    "train_blocks_used": [
        len(lm_train)
    ],

    "validation_blocks_used": [
        len(lm_valid)
    ],

    "train_tokens_used": [
        len(lm_train)
        * BLOCK_SIZE
    ],

    "validation_tokens_used": [
        len(lm_valid)
        * BLOCK_SIZE
    ],

    "initial_val_loss": [
        initial_loss
    ],

    "initial_val_perplexity": [
        initial_perplexity
    ],

    "final_val_loss": [
        final_loss
    ],

    "final_val_perplexity": [
        final_perplexity
    ],

    "loss_improvement": [
        loss_improvement
    ],

    "loss_improvement_percent": [
        loss_improvement_percent
    ],

    "perplexity_improvement": [
        ppl_improvement
    ],

    "perplexity_improvement_percent": [
        ppl_improvement_percent
    ],

    "training_time_hours": [
        training_time / 3600
    ]
})


results_df.to_csv(
    RESULTS_PATH,
    index=False
)


print(
    "\nResults saved to:"
)

print(
    RESULTS_PATH
)



Results saved to:
/kaggle/working/gpt2_legal_dapt_results.csv


In [28]:
print("\n" + "=" * 70)
print("FINAL RESULTS TABLE")
print("=" * 70)


print(
    results_df.to_string(
        index=False
    )
)



FINAL RESULTS TABLE
model architecture initialization                objective  context_length  epochs  learning_rate  weight_decay  per_device_batch_size  gradient_accumulation_steps  effective_batch_size  train_documents  validation_documents  full_train_blocks  full_validation_blocks  train_blocks_used  validation_blocks_used  train_tokens_used  validation_tokens_used  initial_val_loss  initial_val_perplexity  final_val_loss  final_val_perplexity  loss_improvement  loss_improvement_percent  perplexity_improvement  perplexity_improvement_percent  training_time_hours
GPT-2  Transformer     Pretrained Causal Language Modeling            1024       1        0.00002          0.01                      2                            4                     8           120290                 13366             120290                   13366             100000                    2000          102400000                 2048000          3.449754               31.492629        2.789252             

In [29]:
print("\n" + "=" * 70)
print("TESTING DOMAIN-ADAPTED GPT-2")
print("=" * 70)


device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


model = trainer.model

model.to(device)

model.eval()


prompt = (
    "The Supreme Court of India held that"
)


inputs = tokenizer(
    prompt,
    return_tensors="pt"
)


inputs = {

    key: value.to(device)

    for key, value
    in inputs.items()
}


with torch.no_grad():

    generated_ids = model.generate(

        **inputs,

        max_new_tokens=100,

        do_sample=True,

        temperature=0.8,

        top_p=0.95,

        pad_token_id=(
            tokenizer.eos_token_id
        )
    )


generated_text = tokenizer.decode(

    generated_ids[0],

    skip_special_tokens=True
)


print(
    "\nPROMPT:"
)

print(
    prompt
)


print(
    "\nGENERATED TEXT:"
)

print(
    generated_text
)



TESTING DOMAIN-ADAPTED GPT-2

PROMPT:
The Supreme Court of India held that

GENERATED TEXT:
The Supreme Court of India held that the petitioner had no objection to the action of the Board under Section 482 of the Arbitration Act, 1947.The decision of the Supreme Court of India was also not challenged before the High Court and it was held by it that the Act was not as a matter of law and the High Court had no to consider the question as to whether the action of the Board was justified or not.The High Court had to consider the question of the law as to whether the arbitration proceedings were in fact proper and


In [30]:
print("\n" + "=" * 70)
print("EXPERIMENT COMPLETE")
print("=" * 70)


print()
print(
    "GPT-2 LEGAL DAPT SUMMARY"
)

print()
print(
    f"Initial validation loss      : "
    f"{initial_loss:.4f}"
)

print(
    f"Initial validation perplexity: "
    f"{initial_perplexity:.4f}"
)

print()
print(
    f"Final validation loss        : "
    f"{final_loss:.4f}"
)

print(
    f"Final validation perplexity  : "
    f"{final_perplexity:.4f}"
)

print()
print(
    f"Loss improvement             : "
    f"{loss_improvement:.4f}"
)

print(
    f"Loss improvement (%)         : "
    f"{loss_improvement_percent:.2f}%"
)

print()
print(
    f"Perplexity improvement       : "
    f"{ppl_improvement:.4f}"
)

print(
    f"Perplexity improvement (%)   : "
    f"{ppl_improvement_percent:.2f}%"
)

print()
print(
    f"Training time                : "
    f"{training_time / 3600:.2f} hours"
)

print()
print(
    f"Training blocks used         : "
    f"{len(lm_train):,}"
)

print(
    f"Training tokens used         : "
    f"{len(lm_train) * BLOCK_SIZE:,}"
)

print()
print(
    "Final model:"
)

print(
    FINAL_MODEL_DIR
)

print()
print(
    "Results CSV:"
)

print(
    RESULTS_PATH
)

print()
print("=" * 70)


EXPERIMENT COMPLETE

GPT-2 LEGAL DAPT SUMMARY

Initial validation loss      : 3.4498
Initial validation perplexity: 31.4926

Final validation loss        : 2.7893
Final validation perplexity  : 16.2688

Loss improvement             : 0.6605
Loss improvement (%)         : 19.15%

Perplexity improvement       : 15.2238
Perplexity improvement (%)   : 48.34%

Training time                : 6.14 hours

Training blocks used         : 100,000
Training tokens used         : 102,400,000

Final model:
/kaggle/working/gpt2_legal_dapt_final

Results CSV:
/kaggle/working/gpt2_legal_dapt_results.csv

